In [1]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
torch.manual_seed(42)


In [2]:
ATTACK_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\DDoS\\combined_ddos.csv"   # change per attack
BENIGN_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Benign\\combined_benign.csv"
MODEL_PATH   = "ddos_knn.pth" 
SAMPLE_CAP  = 50000
RANDOM_SEED = 42

df_attack = pd.read_csv(ATTACK_FILE)
df_attack = df_attack.sample(n=min(len(df_attack), SAMPLE_CAP), random_state=RANDOM_SEED)

df_benign = pd.read_csv(BENIGN_FILE)
df_benign = df_benign.sample(n=len(df_attack), random_state=RANDOM_SEED)

# Combine and shuffle
df = pd.concat([df_benign, df_attack], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)

X = df.drop(columns=["Label"])
y = df["Label"]

# 60/20/20 split with stratification
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=RANDOM_SEED, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp)

print(f"Attack: {ATTACK_FILE} | Total rows: {len(df)}")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Attack: C:\Users\johfr\Dropbox\Skule\Data science\Bachelor\Dataset\DDoS\combined_ddos.csv | Total rows: 100000
Train: 60000 | Val: 20000 | Test: 20000


In [3]:
from sklearn.preprocessing import StandardScaler

X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [4]:
from sklearn.neighbors import KNeighborsClassifier

k = 5 
knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean", n_jobs=-1)

In [5]:
knn.fit(X_train_scaled, y_train)
print(f"KNN trained with k={k}")

# Use val set to tune k if needed
val_preds = knn.predict(X_val_scaled)
print(classification_report(y_val, val_preds, target_names=["Benign", "Attack"]))

KNN trained with k=5


              precision    recall  f1-score   support

      Benign       0.82      0.90      0.86     10000
      Attack       0.89      0.80      0.84     10000

    accuracy                           0.85     20000
   macro avg       0.85      0.85      0.85     20000
weighted avg       0.85      0.85      0.85     20000



In [6]:
preds = knn.predict(X_test_scaled)

print(classification_report(y_test, preds, target_names=["Benign", "Attack"]))
print(confusion_matrix(y_test, preds))

              precision    recall  f1-score   support

      Benign       0.82      0.91      0.86     10000
      Attack       0.89      0.80      0.85     10000

    accuracy                           0.85     20000
   macro avg       0.86      0.85      0.85     20000
weighted avg       0.86      0.85      0.85     20000

[[9053  947]
 [1957 8043]]
